# Métodos de Análisis de Datos 1
## Unidad 1: Obtención y Preparación de Datos
### Práctica - Clases 3 a 7

**Universidad Nacional del Sur**

---

## Contenido

Este notebook cubre:

1. **Carga de datos** desde diferentes fuentes
2. **Calidad de datos**: valores faltantes, duplicados, inconsistencias
3. **Transformación**: tipos, variables derivadas, codificación
4. **Normalización y estandarización**
5. **Discretización (binning)**
6. **Integración de datos** (joins)
7. **APIs REST**
8. **Tidy data y reshape**

---

## Paso 1: Carga de Datos

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print("Librerías importadas correctamente")

### Crear dataset de ejemplo

Vamos a crear un dataset sintético que simula datos de empleados de una empresa.

In [ ]:
# Crear dataset sintético
np.random.seed(42)
n = 200

# Datos básicos
empleados = pd.DataFrame({
    'id': range(1, n + 1),
    'nombre': [f'Empleado_{i}' for i in range(1, n + 1)],
    'edad': np.random.randint(22, 65, n),
    'departamento': np.random.choice(['IT', 'Ventas', 'RRHH', 'Finanzas', 'Marketing'], n),
    'salario': np.random.randint(30000, 120000, n),
    'fecha_ingreso': pd.date_range('2015-01-01', periods=n, freq='W'),
    'educacion': np.random.choice(['secundaria', 'universitaria', 'posgrado'], n, p=[0.2, 0.6, 0.2])
})

# Introducir problemas de calidad intencionalmente
# 1. Valores faltantes
empleados.loc[np.random.choice(empleados.index, 15, replace=False), 'edad'] = np.nan
empleados.loc[np.random.choice(empleados.index, 10, replace=False), 'salario'] = np.nan

# 2. Duplicados
empleados = pd.concat([empleados, empleados.iloc[[5, 10, 15]]], ignore_index=True)

# 3. Inconsistencias en texto
empleados.loc[20:25, 'departamento'] = empleados.loc[20:25, 'departamento'].str.lower()
empleados.loc[30:32, 'departamento'] = '  IT  '  # Espacios

# 4. Valores fuera de rango
empleados.loc[40, 'edad'] = -5
empleados.loc[41, 'edad'] = 150
empleados.loc[50, 'salario'] = -10000

print(f"Dataset creado: {empleados.shape[0]} filas, {empleados.shape[1]} columnas")
empleados.head(10)

### Exploración inicial

In [ ]:
# Información general
print("Información del dataset:")
empleados.info()

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
empleados.describe()

In [ ]:
# Primeras y últimas filas
print("Primeras 5 filas:")
display(empleados.head())

print("\nÚltimas 5 filas:")
display(empleados.tail())

---

## Paso 2: Calidad de Datos

### 2.1 Valores Faltantes

In [ ]:
# Detectar valores faltantes
print("Valores faltantes por columna:")
missing = empleados.isnull().sum()
missing_pct = (missing / len(empleados)) * 100

missing_df = pd.DataFrame({
    'Total': missing,
    'Porcentaje': missing_pct
})

print(missing_df[missing_df['Total'] > 0])

In [ ]:
# Visualizar patrón de valores faltantes
import missingno as msno

msno.matrix(empleados)
plt.title('Patrón de Valores Faltantes')
plt.show()

### Estrategia de imputación

- **Edad**: Imputar con la mediana (más robusta a outliers)
- **Salario**: Imputar con la media del departamento

In [ ]:
# Copiar dataset para preservar original
df = empleados.copy()

# Imputar edad con mediana
edad_mediana = df['edad'].median()
df['edad'].fillna(edad_mediana, inplace=True)
print(f"Edad: imputado {missing['edad']} valores con mediana = {edad_mediana}")

# Imputar salario con media por departamento
df['salario'] = df.groupby('departamento')['salario'].transform(
    lambda x: x.fillna(x.mean())
)
print(f"Salario: imputado {missing['salario']} valores con media por departamento")

# Verificar
print(f"\nValores faltantes después de imputación: {df.isnull().sum().sum()}")

### 2.2 Duplicados

In [ ]:
# Detectar duplicados
duplicados = df.duplicated()
print(f"Número de filas duplicadas: {duplicados.sum()}")

# Ver duplicados
if duplicados.sum() > 0:
    print("\nFilas duplicadas:")
    display(df[duplicados])

In [ ]:
# Eliminar duplicados
df = df.drop_duplicates()
print(f"Dataset después de eliminar duplicados: {df.shape[0]} filas")

### 2.3 Inconsistencias

In [ ]:
# Verificar valores únicos en departamento
print("Valores únicos en departamento:")
print(df['departamento'].value_counts())

In [ ]:
# Estandarizar texto: lowercase y trim
df['departamento'] = df['departamento'].str.strip().str.upper()

print("\nDespués de estandarización:")
print(df['departamento'].value_counts())

In [ ]:
# Detectar valores fuera de rango
print("Estadísticas de edad:")
print(f"Min: {df['edad'].min()}, Max: {df['edad'].max()}")

print("\nEstadísticas de salario:")
print(f"Min: {df['salario'].min()}, Max: {df['salario'].max()}")

# Detectar valores anormales
edad_invalida = (df['edad'] < 18) | (df['edad'] > 100)
salario_invalido = df['salario'] < 0

print(f"\nEdades inválidas: {edad_invalida.sum()}")
print(f"Salarios inválidos: {salario_invalido.sum()}")

In [ ]:
# Corregir valores fuera de rango
# Opción 1: Reemplazar con NaN y luego imputar
df.loc[df['edad'] < 18, 'edad'] = np.nan
df.loc[df['edad'] > 100, 'edad'] = np.nan
df.loc[df['salario'] < 0, 'salario'] = np.nan

# Imputar nuevamente
df['edad'].fillna(df['edad'].median(), inplace=True)
df['salario'].fillna(df['salario'].mean(), inplace=True)

print("Valores inválidos corregidos")
print(f"Edad - Min: {df['edad'].min()}, Max: {df['edad'].max()}")
print(f"Salario - Min: {df['salario'].min()}, Max: {df['salario'].max()}")

---

## Paso 3: Transformación de Datos

### 3.1 Verificar y convertir tipos

In [ ]:
# Verificar tipos actuales
print("Tipos de datos:")
print(df.dtypes)

In [ ]:
# Convertir tipos
df['id'] = df['id'].astype(str)  # ID como string
df['edad'] = df['edad'].astype(int)  # Edad como entero
df['departamento'] = df['departamento'].astype('category')  # Categórico
df['educacion'] = pd.Categorical(
    df['educacion'],
    categories=['secundaria', 'universitaria', 'posgrado'],
    ordered=True
)  # Categórico ordenado

print("\nTipos después de conversión:")
print(df.dtypes)

### 3.2 Crear variables derivadas

In [ ]:
# Extraer componentes de fecha
df['año_ingreso'] = df['fecha_ingreso'].dt.year
df['mes_ingreso'] = df['fecha_ingreso'].dt.month
df['dia_semana_ingreso'] = df['fecha_ingreso'].dt.dayofweek  # 0=Lunes

# Calcular antigüedad (en años)
hoy = pd.Timestamp.now()
df['antiguedad_años'] = (hoy - df['fecha_ingreso']).dt.days / 365.25

# Crear categorías de edad
df['categoria_edad'] = pd.cut(
    df['edad'],
    bins=[0, 30, 45, 65],
    labels=['joven', 'mediana_edad', 'senior']
)

# Salario por hora (asumiendo 40 hrs/semana, 52 semanas/año)
df['salario_por_hora'] = df['salario'] / (40 * 52)

# Es senior? (>5 años de antigüedad)
df['es_senior'] = df['antiguedad_años'] > 5

print("Variables derivadas creadas:")
print(df[['nombre', 'edad', 'categoria_edad', 'antiguedad_años', 'es_senior']].head())

### 3.3 Codificación de variables categóricas

In [ ]:
# Label Encoding para educación (es ordinal)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['educacion_encoded'] = le.fit_transform(df['educacion'])

print("Label encoding para educación:")
print(dict(zip(le.classes_, le.transform(le.classes_))))
print(df[['educacion', 'educacion_encoded']].head())

In [ ]:
# One-Hot Encoding para departamento (es nominal)
df_encoded = pd.get_dummies(df, columns=['departamento'], prefix='dept')

print("\nOne-hot encoding para departamento:")
print(f"Columnas agregadas: {[col for col in df_encoded.columns if col.startswith('dept_')]}")
print(f"\nDimensiones: {df.shape} → {df_encoded.shape}")

---

## Paso 4: Normalización y Estandarización

In [ ]:
# Antes de escalar, guardar copia
df_pre_scaling = df.copy()

# Seleccionar columnas numéricas para escalar
numeric_cols = ['edad', 'salario', 'antiguedad_años']

### 4.1 Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Aplicar Min-Max
scaler_minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[numeric_cols] = scaler_minmax.fit_transform(df[numeric_cols])

# Comparar
print("Antes de Min-Max:")
print(df[numeric_cols].describe())

print("\nDespués de Min-Max:")
print(df_minmax[numeric_cols].describe())

In [ ]:
# Visualizar distribuciones antes y después
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(numeric_cols):
    # Original
    axes[0, i].hist(df[col], bins=20, edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{col} - Original')
    axes[0, i].set_xlabel('Valor')
    axes[0, i].set_ylabel('Frecuencia')
    
    # Normalizado
    axes[1, i].hist(df_minmax[col], bins=20, edgecolor='black', alpha=0.7, color='orange')
    axes[1, i].set_title(f'{col} - Min-Max [0,1]')
    axes[1, i].set_xlabel('Valor')
    axes[1, i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 4.2 Standardization (Z-score)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Aplicar Standardization
scaler_std = StandardScaler()
df_std = df.copy()
df_std[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])

# Comparar
print("Después de Standardization:")
print(df_std[numeric_cols].describe())
print("\nNota: Media ~0, Std ~1")

---

## Paso 5: Discretización (Binning)

In [ ]:
# Bins de igual ancho
df['salario_bins_ancho'] = pd.cut(
    df['salario'],
    bins=5,
    labels=['muy_bajo', 'bajo', 'medio', 'alto', 'muy_alto']
)

print("Bins de igual ancho (salario):")
print(df['salario_bins_ancho'].value_counts().sort_index())

In [ ]:
# Bins con puntos específicos
df['salario_bins_custom'] = pd.cut(
    df['salario'],
    bins=[0, 40000, 60000, 80000, 120000],
    labels=['entrada', 'junior', 'mid', 'senior']
)

print("\nBins personalizados (salario):")
print(df['salario_bins_custom'].value_counts().sort_index())

In [ ]:
# Bins de igual frecuencia (cuartiles)
df['salario_cuartiles'] = pd.qcut(
    df['salario'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print("\nCuartiles (salario):")
print(df['salario_cuartiles'].value_counts().sort_index())
print("\nNota: Cada cuartil tiene aproximadamente el mismo número de observaciones")

In [ ]:
# Visualizar binning
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['salario_bins_ancho'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Bins de Igual Ancho')
axes[0].set_ylabel('Frecuencia')

df['salario_bins_custom'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Bins Personalizados')
axes[1].set_ylabel('Frecuencia')

df['salario_cuartiles'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Cuartiles (Igual Frecuencia)')
axes[2].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

---

## Paso 6: Integración de Datos

Vamos a crear una segunda tabla (evaluaciones de desempeño) y combinarla con la tabla de empleados.

In [ ]:
# Crear tabla de evaluaciones
np.random.seed(42)

# Seleccionar IDs aleatorios (no todos los empleados tienen evaluación)
ids_evaluados = np.random.choice(df['id'].unique(), size=150, replace=False)

evaluaciones = pd.DataFrame({
    'empleado_id': ids_evaluados,
    'puntaje': np.random.randint(1, 6, 150),  # 1-5
    'fecha_evaluacion': pd.date_range('2024-01-01', periods=150, freq='D'),
    'comentarios': ['Buen desempeño' if np.random.random() > 0.5 else 'Necesita mejorar' for _ in range(150)]
})

print("Tabla de evaluaciones:")
print(evaluaciones.head())
print(f"\nShape: {evaluaciones.shape}")

### 6.1 Inner Join

In [ ]:
# Inner join: solo empleados con evaluación
df_inner = pd.merge(
    df,
    evaluaciones,
    left_on='id',
    right_on='empleado_id',
    how='inner'
)

print(f"Inner join: {df_inner.shape[0]} filas")
print(f"Original empleados: {df.shape[0]}, Evaluaciones: {evaluaciones.shape[0]}")
print(f"\nResultado: solo empleados que tienen evaluación")

### 6.2 Left Join

In [ ]:
# Left join: todos los empleados, con o sin evaluación
df_left = pd.merge(
    df,
    evaluaciones,
    left_on='id',
    right_on='empleado_id',
    how='left'
)

print(f"Left join: {df_left.shape[0]} filas")
print(f"\nEmpleados sin evaluación: {df_left['puntaje'].isnull().sum()}")
print(f"Empleados con evaluación: {df_left['puntaje'].notnull().sum()}")

In [ ]:
# Ver empleados sin evaluación
sin_evaluacion = df_left[df_left['puntaje'].isnull()]
print("Empleados sin evaluación (primeros 5):")
print(sin_evaluacion[['nombre', 'departamento', 'puntaje']].head())

---

## Paso 7: Acceso a APIs REST

Vamos a usar la API pública JSONPlaceholder para practicar.

In [ ]:
import requests

# GET request simple
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)

if response.status_code == 200:
    usuarios = response.json()
    df_usuarios = pd.DataFrame(usuarios)
    print("Usuarios obtenidos de la API:")
    display(df_usuarios[['id', 'name', 'email', 'phone']].head())
else:
    print(f"Error: {response.status_code}")

In [ ]:
# GET con parámetros
url_posts = "https://jsonplaceholder.typicode.com/posts"
params = {'userId': 1}
response = requests.get(url_posts, params=params)

if response.status_code == 200:
    posts = response.json()
    df_posts = pd.DataFrame(posts)
    print(f"\nPosts del usuario 1: {len(df_posts)} posts")
    display(df_posts[['id', 'title']].head())

In [ ]:
# POST request (crear nuevo post)
nuevo_post = {
    'title': 'Mi nuevo post desde Python',
    'body': 'Este es el contenido de mi post',
    'userId': 1
}

response = requests.post(url_posts, json=nuevo_post)

if response.status_code == 201:
    print("\nPost creado exitosamente:")
    print(response.json())
else:
    print(f"Error: {response.status_code}")

---

## Paso 8: Tidy Data y Reshape

### 8.1 Wide → Long (Melt)

In [ ]:
# Crear dataset wide de ejemplo
ventas_wide = pd.DataFrame({
    'producto': ['A', 'B', 'C'],
    '2022': [100, 150, 200],
    '2023': [120, 180, 210],
    '2024': [140, 200, 230]
})

print("Dataset WIDE (cada año es una columna):")
display(ventas_wide)

In [ ]:
# Melt: Wide → Long
ventas_long = pd.melt(
    ventas_wide,
    id_vars=['producto'],
    value_vars=['2022', '2023', '2024'],
    var_name='año',
    value_name='ventas'
)

print("\nDataset LONG (tidy):")
display(ventas_long)

### 8.2 Long → Wide (Pivot)

In [ ]:
# Pivot: Long → Wide
ventas_wide_again = ventas_long.pivot(
    index='producto',
    columns='año',
    values='ventas'
)

# Reset index para tener producto como columna
ventas_wide_again = ventas_wide_again.reset_index()

print("Dataset convertido de LONG a WIDE nuevamente:")
display(ventas_wide_again)

In [ ]:
# Visualizar: Long es mejor para gráficos
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.lineplot(data=ventas_long, x='año', y='ventas', hue='producto', marker='o')
plt.title('Ventas por Producto y Año (desde datos LONG/tidy)')
plt.ylabel('Ventas')
plt.xlabel('Año')
plt.legend(title='Producto')
plt.tight_layout()
plt.show()

---

## Resumen y Conclusiones

En este notebook practicamos:

### Clase 3: Fuentes de Datos y Carga
- ✓ Creación de datasets sintéticos
- ✓ Exploración inicial: `info()`, `describe()`, `head()`

### Clase 4: Calidad de Datos
- ✓ Detección de valores faltantes con `isnull()`
- ✓ Imputación simple (mediana) y por grupo (media por departamento)
- ✓ Detección y eliminación de duplicados con `drop_duplicates()`
- ✓ Estandarización de texto con `str.strip()` y `str.upper()`
- ✓ Corrección de valores fuera de rango

### Clase 5: Transformación
- ✓ Conversión de tipos: `astype()`, `pd.Categorical()`
- ✓ Variables derivadas: extraer componentes de fechas, operaciones aritméticas
- ✓ Label encoding con `LabelEncoder`
- ✓ One-hot encoding con `pd.get_dummies()`

### Clase 6: Normalización e Integración
- ✓ Min-Max scaling con `MinMaxScaler`
- ✓ Standardization con `StandardScaler`
- ✓ Discretización: `pd.cut()` (igual ancho, personalizado), `pd.qcut()` (cuartiles)
- ✓ Joins: `pd.merge()` con `how='inner'` y `how='left'`

### Clase 7: APIs y Tidy Data
- ✓ GET requests con `requests.get()`
- ✓ POST requests con `requests.post()`
- ✓ Wide → Long con `pd.melt()`
- ✓ Long → Wide con `pivot()`

### Buenas Prácticas Aplicadas
- ✓ Semilla fijada para reproducibilidad
- ✓ Copias del dataset antes de transformaciones
- ✓ Verificación en cada paso
- ✓ Visualizaciones para entender transformaciones
- ✓ Código comentado y documentado

---

## Ejercicios Propuestos

1. **Explorar otros datasets:**
   - Descargar un dataset de Kaggle
   - Aplicar todo el pipeline de preparación

2. **APIs:**
   - Usar OpenWeatherMap API (requiere API key gratuita)
   - Obtener datos del clima de varias ciudades
   - Crear DataFrame y analizar

3. **Imputación avanzada:**
   - Probar KNN imputation con `KNNImputer`
   - Comparar resultados con imputación simple

4. **Feature engineering:**
   - Crear más variables derivadas
   - Interacciones entre variables
   - Ratios y proporciones

5. **Integración compleja:**
   - Crear tercera tabla (proyectos)
   - Hacer joins múltiples
   - Analizar datos combinados